# 4.1. Matrix Factorization(MF) 기반 추천

추천을 위한 다양한 알고리즘 분류를 해보면, 크게 메모리 기반 알고리즘과 모델기반 알고리즘으로 나눌 수 있음.

|  | 메모리 기반 알고리즘 | 모델 기반 알고리즘 |
|-------|-------|-------|
| 설명 | 메모리에 있는 데이터를 계산해서 추천하는 방식 | 데이터로부터 미리 모델을 구성 후, 필요시 추천하는 방식 |
| 특징 | 개발 사용자 데이터 집중 | 전체 사용자 패턴 집중 |
| 장점 | 원래 데이터를 충실하게 사용 | 대규모 데이터에 빠르게 반응 |
| 단점 | 대규모 데이터에 느리게 반응 | 모델 생성 과정이 오래 걸림 |
| 예시 | CF | MF, Deep Learning |

MF 알고리즘은 아이템과 유저로 구성된 행렬을 두 개의 행렬로 분해하는 방식임. CF 방식과의 차이가 여기서 드러나는데, CF 방식에서는 아이템-유저 행렬을 full matrix로 활용했었음.

각 아이템과 유저 행렬은 K개의 잠재 요인(latent factor)으로 이루어져 있음.

![MF 알고리즘](../static/img_3.png)

예를 들어, K=2인 경우를 생각해보면 사용자, 아이템에 대한 latent matrix는 아래처럼 구상해볼 수 있음.
- 액션-멜로에 대한 잠재요인 (-1~1)
- 판타지-사실주의에 대한 잠재요인 (-1~1)

# 4.2. SGD(Stochastic Gradient Decent)를 사용한 MF 알고리즘

**MF 알고리즘 개념적 설명**

1. **잠재요인 K 설정**: 도메인에 따라 K가 어느정도가 좋을지 결정할 수도 있겠지만, 일반적으로 여러 개의 K 값을 할당해서 비교해가면서 실험하여, 최적의 K를 찾아냄.
2. **P,Q 행렬 초기화**
3. **예측 평점 R_hat 계산**: R_hat= P x Q^T
4. **실제 R과 R_hat간 오차 계산 및 P,Q 수정**: 오차를 줄이는 방향으로 P,Q를 수정하는 것이 MF의 핵심
5. **기준 오차 도달 확인**: 결과에 따라 다시 3번부터 수행

**SGD를 사용하는 이유**

P, Q를 한 번에 정확히 구하는 건 어려움 & 평점 데이터는 희소(sparse)

👉 관측된 평점만 하나씩 보면서 조금씩 고치는 것이 SGD 방식.

**SGD 기반 MF 학습 과정**

실제 평점과 예측 평점의 차이를 최소화

- 예측 평점
$$
\hat{r}_{ui} = p_u^{T} q_i
$$

- 오차
$$
e_{ui} = r_{ui} - \hat{r}_{ui}
$$

- 파라미터 업데이트 
$$
\begin{aligned}
p_u &\leftarrow p_u
+ \alpha ( e_{ui} q_i - \lambda p_u ) \\
q_i &\leftarrow q_i
+ \alpha ( e_{ui} p_u - \lambda q_i )
\end{aligned}
$$
$$
\alpha : \text{learning rate}, \quad
\lambda : \text{regularization coefficient (to prevent overfitting)}
$$


**Overfitting이 생기는 이유**

직관적으로
- 평점 데이터는 적은데
- 잠재 요인은 많음

👉 모델이 훈련 데이터만 외워버림

이 경우, 모델이 특정 사용자-아이템 조합은 잘 맞히지만, 새로운 데이터에는 엉망인 결과를 가져올 수 있음.

Overfitting을 방지(Regularization)하기 위해 등장하는 것이 추가 계수 $\lambda$

- 손실 함수: 원래 항에 추가적인 항을 붙인 형태가 됨
$$
\mathcal{L} =
\sum_{(u,i)\in\mathcal{D}}
\left(r_{ui} - p_u^{T} q_i\right)^2
+
\lambda
\left(
\|p_u\|^2 + \|q_i\|^2
\right)
$$

$\lambda$항은 파라미터를 작고 부드럽게 유지해주는 역할. 

그래서 SGD의 업데이트 과정을 보면, $\lambda$항이 작용하고 있음을 볼 수 있음.

$$
p_u \leftarrow p_u
+ \alpha
\left(
e_{ui} q_i - \lambda p_u
\right)
$$

$\lambda$가 너무 작으면 Overfitting으로 이어질 수 있고, 너무 크면 Underfitting으로 이어질 수 있기에 적절한 크기로 상정하는 것이 중요.